In [0]:
dbutils.widgets.text("Source_System", "", "Source System")
dbutils.widgets.text("Table_Name", "", "Table Name")
dbutils.widgets.text("Landing_Timestamp", "", "Landing Timestamp")
dbutils.widgets.text("Source_File_Format", "", "Source File Format")

source_system = dbutils.widgets.get("Source_System")
table_name = dbutils.widgets.get("Table_Name")
landing_timestamp = dbutils.widgets.get("Landing_Timestamp")
source_file_format = dbutils.widgets.get("Source_File_Format")

In [0]:
path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

try:
    files = dbutils.fs.ls(path)
except Exception:
    files = []

if not files:
    msg = f"No new files are received at {path}. Exiting notebook."
    print(msg)
    dbutils.notebook.exit(msg)


In [0]:
if source_file_format == "csv":
    df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(path)
else:
    df = spark.read.format("parquet").load(path)


In [0]:
from pyspark.sql.functions import lit, current_timestamp

df = df.withColumn("landing_timestamp", lit(landing_timestamp)) \
       .withColumn("ingestion_timestamp", current_timestamp())



In [0]:
df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")